In [2]:
import random
from IPython.display import clear_output
random.seed(42)

class MultiLevelEnv:
    def __init__(self, num_levels=10, level_width=10):
        self.num_levels = num_levels
        self.level_width = level_width
        # Action Map: 0=Left, 1=Right, 2=Up, 3=Down
        self.action_space = [0, 1, 2, 3] 
        
        self.holes = {}
        self._generate_holes()
        self.reset()

    def _generate_holes(self):
        """Internal: Generates 1 to 3 holes per intermediate level."""
        for lvl in range(self.num_levels - 1):
            max_possible_holes = min(3, self.level_width)
            num_holes = random.randint(1, max_possible_holes)
            self.holes[lvl] = random.sample(range(self.level_width), num_holes)

    def reset(self, random_gen=False):
        """
        Resets the agent state.
        :param random_gen: If True, agent starts at a random level and position.
                           If False, agent starts at (0, 0).
        """
        if random_gen:
            # Random level from 0 to the second-to-top level 
            # (Starting exactly on the goal level 9 would end the game immediately)
            self.agent_level = random.randint(0, self.num_levels - 2)
            self.agent_pos = random.randint(0, self.level_width - 1)
        else:
            self.agent_level = 0
            self.agent_pos = 0
            
        return (self.agent_level, self.agent_pos)

    def get_action_space(self):
        """Returns the full list of actions available in the environment."""
        return self.action_space

    def step(self, action):
        """
        Executes an action.
        Returns: next_state (tuple), reward (int), done (bool), info (str)
        """
        lvl, pos = self.agent_level, self.agent_pos
        reward = 0
        done = False
        info = "Move Successful"

        if action == 0:   # LEFT
            if pos > 0:
                pos -= 1
            else:
                done = True
                reward = -70
                info = "Terminated: Hit Left Wall"
            
        elif action == 1: # RIGHT
            if pos < self.level_width - 1:
                pos += 1
            else:
                done = True
                reward = -70 
                info = "Terminated: Hit Right Wall"
            
        elif action == 2: # UP
            if lvl < self.num_levels - 1:
                if pos in self.holes.get(lvl, []):
                    lvl += 1
                    reward = 10 
                else:
                    done = True
                    reward = -50
                    info = "Terminated: Hit Ceiling (No Hole)"
            else:
                done = True
                reward = -50
                info = "Terminated: Already at Top Level" # no need to change this reward

        elif action == 3: # DOWN
            if lvl > 0:
                if pos in self.holes.get(lvl - 1, []):
                    lvl -= 1
                    reward =-100
                else:
                    done = True
                    reward = -50
                    info = "Terminated: Hit Floor (No Hole)"
            else:
                done = True
                reward = -50
                info = "Terminated: Cannot go below Level 0" # no need to change this reward

        # Update state
        self.agent_level, self.agent_pos = lvl, pos

        # Goal Check
        if not done and self.agent_level == self.num_levels - 1:
            # reward = 
            done = True
            info = "Goal Reached!"

        return (self.agent_level, self.agent_pos), reward, done, info

    def render(self):
        """Prints the environment. Uses brackets to ensure perfect alignment."""
        header = "        " + " === " * self.level_width
        print("\n" + header)
        
        for lvl in range(self.num_levels - 1, -1, -1):
            row_str = [f"Lvl {lvl:>2}: |"] 
            
            for pos in range(self.level_width):
                if self.agent_level == lvl and self.agent_pos == pos:
                    row_str.append("[🤖]") 
                elif lvl == self.num_levels - 1:
                    row_str.append("[🏆]") 
                elif lvl < self.num_levels - 1 and pos in self.holes[lvl]:
                    row_str.append("[  ]") 
                else:
                    row_str.append("[==]") 
            
            row_str.append("|")
            print("".join(row_str))
            
        print(header)
        print(f"Current State: (Level {self.agent_level}, Pos {self.agent_pos})\n")
# Initialize
env = MultiLevelEnv(num_levels=20, level_width=10)

In [3]:
state = env.reset()
env.render()


         ===  ===  ===  ===  ===  ===  ===  ===  ===  === 
Lvl 19: |[🏆][🏆][🏆][🏆][🏆][🏆][🏆][🏆][🏆][🏆]|
Lvl 18: |[==][==][==][  ][  ][  ][==][==][==][==]|
Lvl 17: |[==][  ][==][==][==][==][  ][==][==][  ]|
Lvl 16: |[==][==][==][==][==][==][==][  ][==][==]|
Lvl 15: |[==][==][==][==][  ][==][==][==][==][  ]|
Lvl 14: |[==][  ][==][==][==][  ][==][==][==][==]|
Lvl 13: |[==][  ][==][==][==][==][==][==][==][  ]|
Lvl 12: |[==][==][==][  ][==][==][==][==][==][==]|
Lvl 11: |[==][==][==][==][  ][  ][  ][==][==][==]|
Lvl 10: |[==][==][  ][==][==][==][==][==][==][==]|
Lvl  9: |[==][==][==][==][  ][==][==][==][==][  ]|
Lvl  8: |[==][==][==][  ][==][==][  ][==][  ][==]|
Lvl  7: |[  ][==][==][  ][==][==][==][==][  ][==]|
Lvl  6: |[==][==][==][==][==][==][==][==][  ][==]|
Lvl  5: |[==][==][==][  ][==][==][==][==][==][==]|
Lvl  4: |[  ][==][==][==][==][==][==][==][==][==]|
Lvl  3: |[==][  ][==][==][==][==][  ][==][  ][==]|
Lvl  2: |[==][  ][==][==][==][==][==][==][==][==]|
Lvl  1: |[==][==][==][  ][==][==

In [4]:
# print(state, reward, done, info)
# state, reward, done, info = env.step(1)
# print(f"Game Over! Reason: {info}",done)
# print(state, reward, done, info)
# # state, reward, done, info = env.step(1)
# # print(f"Game Over! Reason: {info}",done)
# # # state, reward, done, info = env.step(1)
# env.render()
# # print(f"Game Over! Reason: {info}",done)

In [5]:
env.num_levels*env.level_width

200

In [6]:

state

(0, 0)

In [7]:
env.agent_level,env.agent_pos,4

(0, 0, 4)

In [8]:
import numpy as np

In [20]:
gamma=0.9

Q=np.zeros((env.num_levels*env.level_width,4))
def lokesh_sir(env):

    for i in range(1000000):
        b=[0,1,2,3]
        
        curr_s = env.agent_level*10+env.agent_pos
        
        if random.uniform(0,1) > 0.8:
            action=random.choice(b)
        else:
            action = np.argmax(Q[curr_s])
        
        
        state, reward, done, info = env.step(action)
        index = state[0] *10 + state[1]
        Q[curr_s,action] = reward+gamma*max(Q[index])
        
        if done:
            env.reset(True)
        else :
            continue 
    return Q


In [21]:
lokesh_sir(env)

array([[-36.77974226,  35.91139749,  36.91139749, -16.77974226],
       [ 33.22025774,  32.32025774,  39.90155277, -14.08860251],
       [ 35.91139749,  35.01139749, -17.67974226, -17.67974226],
       [ 32.32025774,  38.90155277, -14.98860251, -14.98860251],
       [ 35.01139749,  35.01139749,  43.22394752, -11.09844723],
       [ 38.90155277,  31.51025774, -14.98860251, -14.98860251],
       [ 35.01139749,  28.35923197, -18.48974226, -18.48974226],
       [ 31.51025774,  25.52330877, -21.64076803, -21.64076803],
       [ 28.35923197,  22.97097789, -24.47669123, -24.47669123],
       [ 25.52330877, -47.02902211, -27.02902211, -27.02902211],
       [-43.08860251,  29.90155277, -23.08860251, -66.77974226],
       [ 26.91139749,  33.22394752, -20.09844723, -64.08860251],
       [ 29.90155277,  36.91549724, -16.77605248, -16.77605248],
       [ 33.22394752,  33.22394752,  41.01721916, -13.08450276],
       [ 36.91549724,  29.90155277, -16.77605248, -61.09844723],
       [ 33.22394752,  26

In [22]:
import numpy as np

def play_with_q_table(env, q_table, render=True, max_steps=1000):
    """
    Plays the game using a Q-table matrix.
    
    Args:
        env: The MultiLevelEnv instance.
        q_table: A 2D numpy array of shape (num_states, 4).
        render: Whether to show the agent's movement.
        max_steps: A limit to prevent infinite loops if the agent is stuck.
        
    Returns:
        steps: Total steps taken to reach the goal. 
               Returns 9999 if the agent dies or fails.
    """
    state = env.reset()
    done = False
    total_steps = 0
    
    if render:
        print("Starting playback with Q-Table...")
        env.render()

    while not done and total_steps < max_steps:
        # 1. Convert tuple state (lvl, pos) to matrix index
        lvl, pos = state
        state_idx = lvl * env.level_width + pos
        
        # 2. Select the best action (greedy) from the matrix row
        action = np.argmax(q_table[state_idx])
        
        # 3. Take the step
        state, reward, done, info = env.step(action)
        total_steps += 1
        
        if render:
            env.render()

            print(f"Step {total_steps}: Moved {['Left', 'Right', 'Up', 'Down'][action]} | Info: {info}")

        # 4. Check for success or failure
        if done:
            if info == "Goal Reached!":
                if render: print(f"✅ Success! Goal reached in {total_steps} steps.")
                return total_steps
            else:
                if render: print(f"❌ Failed! Reason: {info}")
                return 9999 # High penalty score for failing
        clear_output(wait=True)
    
    if render: print("⌛ Failure: Maximum steps reached without reaching the goal.")
    return 9999
    time.sleep(0.5)

# --- Example of how the student would generate the Q-table shape ---
# num_states = env.num_levels * env.level_width
# q_table = np.zeros((num_states, 4))
play_with_q_table(env, Q, render=True, max_steps=1000)


         ===  ===  ===  ===  ===  ===  ===  ===  ===  === 
Lvl 19: |[🏆][🏆][🏆][🏆][🏆][🤖][🏆][🏆][🏆][🏆]|
Lvl 18: |[==][==][==][  ][  ][  ][==][==][==][==]|
Lvl 17: |[==][  ][==][==][==][==][  ][==][==][  ]|
Lvl 16: |[==][==][==][==][==][==][==][  ][==][==]|
Lvl 15: |[==][==][==][==][  ][==][==][==][==][  ]|
Lvl 14: |[==][  ][==][==][==][  ][==][==][==][==]|
Lvl 13: |[==][  ][==][==][==][==][==][==][==][  ]|
Lvl 12: |[==][==][==][  ][==][==][==][==][==][==]|
Lvl 11: |[==][==][==][==][  ][  ][  ][==][==][==]|
Lvl 10: |[==][==][  ][==][==][==][==][==][==][==]|
Lvl  9: |[==][==][==][==][  ][==][==][==][==][  ]|
Lvl  8: |[==][==][==][  ][==][==][  ][==][  ][==]|
Lvl  7: |[  ][==][==][  ][==][==][==][==][  ][==]|
Lvl  6: |[==][==][==][==][==][==][==][==][  ][==]|
Lvl  5: |[==][==][==][  ][==][==][==][==][==][==]|
Lvl  4: |[  ][==][==][==][==][==][==][==][==][==]|
Lvl  3: |[==][  ][==][==][==][==][  ][==][  ][==]|
Lvl  2: |[==][  ][==][==][==][==][==][==][==][==]|
Lvl  1: |[==][==][==][  ][==][==

52

In [19]:
import time
for _ in range(100):
    
    curr_s = env.agent_level*10+env.agent_pos
    action = np.argmax(Q[curr_s])
    state, reward, done, info = env.step(action)
    env.render()
    clear_output(wait=True)
    time.sleep(0.5)
    if done:
        break

KeyboardInterrupt: 